# Parse FIX
Resolve classified market message rows against the FIX dictionary.

In [ ]:
project_root = "."
source = "logs.messages"
start = None
end = None
fix_dictionary = "data/fix"
null_values = ["", "null", "<null>", "n/a", "none"]
protocols = None
fields = None
catalog = "rekep"
catalog_properties = {}
table_properties = {"history.expire.max-snapshot-age-ms": "604800000"}
branch = "root"
target_pattern = "fix.{category}"
merge_by = True
commit_row_size = 250_000
limit = None
log_level = "INFO"

In [ ]:
import pyarrow
import pyarrow.compute as pc
from pyiceberg.expressions import (
    And,
    GreaterThanOrEqual,
    In,
    LessThan,
    Not,
)

from rekep.enums import EventType
from rekep.fix.fields import FieldRules
from rekep.fix.registry import FixRegistry
from rekep.fix.rules import Rules
from rekep.fix.transcribe import FixCodec
from rekep.iceberg import IcebergDataset
from rekep.logs import Stage, configure
from rekep.text import FixMsg, Message
from rekep.times import unix_of
from rekep.urls import Url

configure(log_level)
if commit_row_size <= 0:
    raise ValueError("commit_row_size must be positive")


def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


protocol_rules = Rules() if protocols is None else Rules.from_dict(protocols)
registry = FixRegistry(
    cache_dir=Url.from_string(str(fix_dictionary)).resolve(project_root),
    announce=print,
)
field_rules = FieldRules() if fields is None else FieldRules.from_dict(fields)
codec = FixCodec(
    rules=protocol_rules,
    registry=registry,
    null_values=frozenset(null_values),
    fields=field_rules,
)
field = FixMsg.into_field()
source_field = Message.into_field()
messages = IcebergDataset(
    field=source_field.with_name(source),
    catalog=catalog,
    properties=dict(catalog_properties),
    branch=branch,
)
stage = Stage(
    "parse_fix",
    sources={"messages": source},
    window=(unix_of(start), unix_of(end, upper=True)),
)
source_columns = list(
    messages.table_field.names if messages.exists else source_field.names
)
if messages.exists:
    missing = sorted({"msgtype", "entries", "protocol"} - set(messages.table_field.names))
    if missing:
        raise ValueError(
            f"{source} is missing {missing}; rebuild it with parse_messages before parsing FIX"
        )

In [ ]:
read = market_read = errors = 0
# What the stages after this one key on. This one owns resolving the
# transaction clock and nesting the Instrument whose class maps the canonical
# ticker, so it also says how well it managed: which rung answered for `unix`
# on each row, how many carry an `instrument.symbolticker`, and how many were
# retained with a row-local transcription error. A run
# that hands on a weak base says so here instead of two tables later.
unixsource = {}
tickered = 0


def _measured(batch):
    """One parsed batch, with its clock and ticker coverage counted."""
    global tickered, errors
    for source, count in zip(
        *(column.to_pylist() for column in pc.value_counts(batch.column("unixsource")).flatten()),
        strict=True,
    ):
        unixsource[source] = unixsource.get(source, 0) + count
    symbolticker = pc.struct_field(batch.column("instrument"), "symbolticker")
    tickered += pc.sum(pc.not_equal(symbolticker, ""), min_count=0).as_py() or 0
    errors += pc.sum(pc.is_valid(batch.column("error")), min_count=0).as_py() or 0
    return batch


# The window is read off the *stored* recording clock, because that is what
# the message stage partitioned on. `unix` moves when a transaction time
# resolves, so filtering on it here would drop rows the interval owns.
lower, upper = unix_of(start), unix_of(end, upper=True)
window = _window(lower, upper)
# The two scans partition every stored `eventtype`: the market scan keeps the
# compiled codes ranked at or above INTENT, and the terminal scan keeps the
# complement. A code no member spells still reaches the category router instead
# of silently matching no scan.
market_events = In("eventtype", EventType.ranked_at_least(EventType.INTENT))
terminal_events = Not(market_events)
market_filter = market_events if window is None else And(window, market_events)
terminal_filter = terminal_events if window is None else And(window, terminal_events)
targets = {}


def _target(category):
    target = targets.get(category)
    if target is None:
        target = targets[category] = IcebergDataset(
            field=field.with_name(target_pattern.format(category=category)),
            catalog=catalog,
            properties=dict(catalog_properties),
            table_properties=dict(table_properties),
            branch=branch,
            commit_row_size=commit_row_size,
        )
        stage.targets[category] = target.name
    return target


target = _target("market")


def _market_batches():
    global read, market_read
    for staged in messages.read_arrow_reader(columns=source_columns, row_filter=market_filter):
        if limit is not None and read + staged.num_rows > limit:
            staged = staged.slice(0, max(0, limit - read))
        if not staged.num_rows:
            continue
        read += staged.num_rows
        market_read += staged.num_rows
        batch = _measured(FixMsg.from_message_batch(staged, codec))
        yield batch
        if limit is not None and read >= limit:
            break


written = target.append_arrow_reader(
    _market_batches(),
    field,
    merge_by=merge_by,
    commit_row_size=commit_row_size,
)
skipped = market_read - written
routed = {"market": market_read}
buffers = {}
held_rows = {}


def _flush(category):
    global written, skipped
    batches = buffers.pop(category, [])
    count = held_rows.pop(category, 0)
    if not count:
        return
    landed = _target(category).append_arrow_table(
        pyarrow.Table.from_batches(batches), merge_by=merge_by
    )
    written += landed
    skipped += count - landed


for staged in messages.read_arrow_reader(columns=source_columns, row_filter=terminal_filter):
    if limit is not None and read >= limit:
        break
    if limit is not None and read + staged.num_rows > limit:
        staged = staged.slice(0, max(0, limit - read))
    if not staged.num_rows:
        continue
    read += staged.num_rows
    batch = _measured(FixMsg.from_message_batch(staged, codec))
    categories = protocol_rules.into_arrow_category_array(
        batch.column("protocol"), batch.column("eventtype")
    )
    for category in sorted(pc.unique(categories).to_pylist()):
        part = batch.filter(pc.equal(categories, category))
        routed[category] = routed.get(category, 0) + part.num_rows
        buffers.setdefault(category, []).append(part)
        held_rows[category] = held_rows.get(category, 0) + part.num_rows
        if held_rows[category] >= commit_row_size:
            _flush(category)
    if limit is not None and read >= limit:
        break
for category in list(buffers):
    _flush(category)

In [ ]:
stage.says(
    "routed %s", ", ".join(f"{count} {category}" for category, count in sorted(routed.items()))
)
# The base the next two stages key on, said where it was built rather than
# discovered two tables later.
stage.says(
    "resolved unix from %s; %d of %d rows carry a symbolticker",
    ", ".join(f"{rung} {count}" for rung, count in sorted(unixsource.items())) or "nothing",
    tickered,
    read,
)
stage.says("retained %d rows with FIX transcription errors", errors)
result = stage.finished(
    read=read,
    written=written,
    skipped=skipped,
    routed=routed,
    unixsource=unixsource,
    tickered=tickered,
    errors=errors,
)
result